# DBMS Interview Questions - With SQL Code & Output

Each question has the answer, a runnable SQL example (using SQLite, no setup needed) and its output. Open in Google Colab and click **Runtime → Run all**. Questions 1 and 2 are theory only.

## Setup
Run this first. It creates an in-memory database and the sample `Employees` and `Departments` tables used in the examples.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON")   # SQLite needs this to enforce foreign keys

def run(query):
    """Run a SELECT query and show the result as a table."""
    return pd.read_sql_query(query, conn)

def execute(script):
    """Run statements that change data or structure."""
    conn.executescript(script)

In [2]:
execute("""
DROP TABLE IF EXISTS Employees;
DROP TABLE IF EXISTS Departments;
CREATE TABLE Departments (dept_id INTEGER PRIMARY KEY, dept_name TEXT);
CREATE TABLE Employees (
    id INTEGER PRIMARY KEY, name TEXT, salary REAL, dept_id INTEGER,
    FOREIGN KEY (dept_id) REFERENCES Departments(dept_id)
);
INSERT INTO Departments VALUES (1,'CSE'),(2,'ECE'),(3,'EEE'),(4,'MECH');
INSERT INTO Employees VALUES
(1,'Asha',50000,1),(2,'Ravi',45000,2),(3,'Priya',60000,1),
(4,'Arun',40000,3),(5,'Meena',60000,1),(6,'Kiran',30000,NULL);
""")
run("SELECT * FROM Employees")

,id,name,salary,dept_id
0,1,Asha,50000.0,1.0
1,2,Ravi,45000.0,2.0
2,3,Priya,60000.0,1.0
3,4,Arun,40000.0,3.0
4,5,Meena,60000.0,1.0
5,6,Kiran,30000.0,NaN


## 1. 1. What is DBMS?

DBMS (Database Management System) is software used to create, store, manage, and retrieve data from databases.

## 2. 2. What is RDBMS?

RDBMS stores data in tables and generally uses relationships between tables. SQL is commonly used to work with relational databases.

## 3. 3. What is a primary key?

A primary key uniquely identifies each record in a table. It cannot contain duplicate values and cannot normally be NULL.

In [3]:
execute("DROP TABLE IF EXISTS Students; CREATE TABLE Students (id INTEGER PRIMARY KEY, name TEXT)")
conn.execute("INSERT INTO Students VALUES (1, 'Asha')")
try:
    conn.execute("INSERT INTO Students VALUES (1, 'Ravi')")   # duplicate key
except sqlite3.IntegrityError as e:
    print("Duplicate primary key rejected ->", e)
run("SELECT * FROM Students")

Duplicate primary key rejected -> UNIQUE constraint failed: Students.id


,id,name
0,1,Asha


## 4. 4. What is a foreign key?

A foreign key is a column or set of columns that refers to a key in another table and helps establish relationships between tables.

In [4]:
# Employees.dept_id is a foreign key referring to Departments.dept_id
try:
    conn.execute("INSERT INTO Employees VALUES (99, 'Test', 10000, 77)")   # dept 77 does not exist
except sqlite3.IntegrityError as e:
    print("Invalid foreign key rejected ->", e)

Invalid foreign key rejected -> FOREIGN KEY constraint failed


## 5. 5. What is a candidate key?

A candidate key is a minimal set of attributes that can uniquely identify a record.

In [5]:
# In Employees, 'id' is a candidate key. In a table with email, both id and email could be.
execute("""DROP TABLE IF EXISTS Users;
CREATE TABLE Users (id INTEGER PRIMARY KEY, email TEXT UNIQUE, name TEXT);
INSERT INTO Users VALUES (1,'a@x.com','Asha'),(2,'r@x.com','Ravi');""")
print("Candidate keys here: id, email (each uniquely identifies a row)")
run("SELECT * FROM Users")

Candidate keys here: id, email (each uniquely identifies a row)


,id,email,name
0,1,a@x.com,Asha
1,2,r@x.com,Ravi


## 6. 6. What is a super key?

A super key is a set of one or more attributes that can uniquely identify a record.

In [6]:
# Super keys of Users: any set containing a candidate key, e.g. {id}, {email}, {id, name}, {id, email}
run("SELECT id, name FROM Users")   # {id, name} is unique too, so it is a super key (not minimal)

,id,name
0,1,Asha
1,2,Ravi


## 7. 7. What are constraints?

Constraints are rules applied to table columns. Common constraints include PRIMARY KEY, FOREIGN KEY, UNIQUE, NOT NULL, CHECK, and DEFAULT.

In [7]:
execute("""DROP TABLE IF EXISTS Products;
CREATE TABLE Products (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    code TEXT UNIQUE,
    price REAL CHECK (price > 0),
    status TEXT DEFAULT 'active'
);""")
conn.execute("INSERT INTO Products (id, name, code, price) VALUES (1, 'Pen', 'P1', 10)")

tests = {
  "NOT NULL": "INSERT INTO Products (id, name, code, price) VALUES (2, NULL, 'P2', 5)",
  "UNIQUE"  : "INSERT INTO Products (id, name, code, price) VALUES (3, 'Book', 'P1', 5)",
  "CHECK"   : "INSERT INTO Products (id, name, code, price) VALUES (4, 'Bag', 'P4', -5)",
}
for name, sql in tests.items():
    try:
        conn.execute(sql)
    except sqlite3.IntegrityError as e:
        print(f"{name:9} rejected -> {e}")
print("\nDEFAULT applied:")
run("SELECT * FROM Products")

NOT NULL  rejected -> NOT NULL constraint failed: Products.name
UNIQUE    rejected -> UNIQUE constraint failed: Products.code
CHECK     rejected -> CHECK constraint failed: price > 0

DEFAULT applied:


,id,name,code,price,status
0,1,Pen,P1,10.0,active


## 8. 8. What is normalization?

Normalization organizes data to reduce redundancy and improve data integrity.

In [8]:
# Unnormalized: department details repeated in every row
execute("""DROP TABLE IF EXISTS Unnormalized;
CREATE TABLE Unnormalized (emp TEXT, dept_name TEXT, dept_head TEXT);
INSERT INTO Unnormalized VALUES ('Asha','CSE','Dr. Rao'),('Priya','CSE','Dr. Rao'),('Ravi','ECE','Dr. Sen');""")
print("Before normalization (CSE and Dr. Rao repeated):")
display(run("SELECT * FROM Unnormalized"))

# Normalized: split into two tables
execute("""DROP TABLE IF EXISTS Dept; DROP TABLE IF EXISTS Emp;
CREATE TABLE Dept (dept_id INTEGER PRIMARY KEY, dept_name TEXT, dept_head TEXT);
CREATE TABLE Emp (emp TEXT, dept_id INTEGER);
INSERT INTO Dept VALUES (1,'CSE','Dr. Rao'),(2,'ECE','Dr. Sen');
INSERT INTO Emp VALUES ('Asha',1),('Priya',1),('Ravi',2);""")
print("After normalization:")
display(run("SELECT * FROM Dept"))
run("SELECT * FROM Emp")

Before normalization (CSE and Dr. Rao repeated):


,emp,dept_name,dept_head
0,Asha,CSE,Dr. Rao
1,Priya,CSE,Dr. Rao
2,Ravi,ECE,Dr. Sen


After normalization:


,dept_id,dept_name,dept_head
0,1,CSE,Dr. Rao
1,2,ECE,Dr. Sen


,emp,dept_id
0,Asha,1
1,Priya,1
2,Ravi,2


## 9. 9. What are 1NF, 2NF, and 3NF?

1NF requires atomic values. 2NF requires 1NF and removal of partial dependency. 3NF requires 2NF and removal of transitive dependency.

In [9]:
print("1NF violation: a cell holding several values")
display(pd.DataFrame({"student": ["Asha"], "phones": ["9999, 8888"]}))
print("1NF fix: one value per cell")
display(pd.DataFrame({"student": ["Asha", "Asha"], "phone": ["9999", "8888"]}))
print("2NF: (student_id, course_id) -> student_name is a partial dependency; move student_name to a Students table.")
print("3NF: emp_id -> dept_id -> dept_name is transitive; move dept_name to a Departments table.")

1NF violation: a cell holding several values


,student,phones
0,Asha,"9999, 8888"


1NF fix: one value per cell


,student,phone
0,Asha,9999
1,Asha,8888


2NF: (student_id, course_id) -> student_name is a partial dependency; move student_name to a Students table.
3NF: emp_id -> dept_id -> dept_name is transitive; move dept_name to a Departments table.


## 10. 10. What is SQL?

SQL (Structured Query Language) is used to create, query, modify, and manage data in relational databases.

In [10]:
print("SQL is used for statements like:")
run("SELECT name, salary FROM Employees WHERE salary > 45000 ORDER BY salary DESC")

SQL is used for statements like:


,name,salary
0,Priya,60000.0
1,Meena,60000.0
2,Asha,50000.0


## 11. 11. What are DDL, DML, DCL, and TCL?

- DDL: CREATE, ALTER, DROP, TRUNCATE
- DML: INSERT, UPDATE, DELETE
- DCL: GRANT, REVOKE
- TCL: COMMIT, ROLLBACK, SAVEPOINT

In [11]:
# DDL
execute("DROP TABLE IF EXISTS Demo; CREATE TABLE Demo (id INTEGER, name TEXT)")
execute("ALTER TABLE Demo ADD COLUMN age INTEGER")
print("DDL: table created and altered")

# DML
conn.execute("INSERT INTO Demo VALUES (1,'Asha',20)")
conn.execute("INSERT INTO Demo VALUES (2,'Ravi',21)")
conn.execute("UPDATE Demo SET age = 22 WHERE id = 2")
conn.execute("DELETE FROM Demo WHERE id = 1")
print("DML: insert / update / delete done")
display(run("SELECT * FROM Demo"))

# TCL
conn.commit()
conn.execute("INSERT INTO Demo VALUES (3,'Temp',30)")
conn.rollback()
print("TCL: commit saved earlier changes, rollback undid the 'Temp' row")
display(run("SELECT * FROM Demo"))
print("DCL (GRANT/REVOKE) is not supported by SQLite; it is used in MySQL, PostgreSQL, Oracle etc.")

DDL: table created and altered
DML: insert / update / delete done


,id,name,age
0,2,Ravi,22


TCL: commit saved earlier changes, rollback undid the 'Temp' row


,id,name,age
0,2,Ravi,22


DCL (GRANT/REVOKE) is not supported by SQLite; it is used in MySQL, PostgreSQL, Oracle etc.


## 12. 12. What is a JOIN?

A JOIN combines rows from two or more tables based on a related column.

In [12]:
run("""
SELECT e.name, d.dept_name
FROM Employees e
JOIN Departments d ON e.dept_id = d.dept_id
""")

,name,dept_name
0,Asha,CSE
1,Ravi,ECE
2,Priya,CSE
3,Arun,EEE
4,Meena,CSE


## 13. 13. Types of JOINs

Common joins include INNER JOIN, LEFT JOIN, RIGHT JOIN, and FULL OUTER JOIN.

In [13]:
print("INNER JOIN - only matching rows")
display(run("SELECT e.name, d.dept_name FROM Employees e INNER JOIN Departments d ON e.dept_id = d.dept_id"))

print("LEFT JOIN - all employees, even Kiran with no department")
display(run("SELECT e.name, d.dept_name FROM Employees e LEFT JOIN Departments d ON e.dept_id = d.dept_id"))

print("RIGHT JOIN - all departments, even MECH with no employees (written as a LEFT JOIN with tables swapped)")
display(run("SELECT e.name, d.dept_name FROM Departments d LEFT JOIN Employees e ON e.dept_id = d.dept_id"))

print("FULL OUTER JOIN - everything from both (LEFT JOIN UNION RIGHT JOIN)")
run("""
SELECT e.name, d.dept_name FROM Employees e LEFT JOIN Departments d ON e.dept_id = d.dept_id
UNION
SELECT e.name, d.dept_name FROM Departments d LEFT JOIN Employees e ON e.dept_id = d.dept_id
""")

INNER JOIN - only matching rows


,name,dept_name
0,Asha,CSE
1,Ravi,ECE
2,Priya,CSE
3,Arun,EEE
4,Meena,CSE


LEFT JOIN - all employees, even Kiran with no department


,name,dept_name
0,Asha,CSE
1,Ravi,ECE
2,Priya,CSE
3,Arun,EEE
4,Meena,CSE
5,Kiran,NaN


RIGHT JOIN - all departments, even MECH with no employees (written as a LEFT JOIN with tables swapped)


,name,dept_name
0,Asha,CSE
1,Meena,CSE
2,Priya,CSE
3,Ravi,ECE
4,Arun,EEE
5,NaN,MECH


FULL OUTER JOIN - everything from both (LEFT JOIN UNION RIGHT JOIN)


,name,dept_name
0,NaN,MECH
1,Arun,EEE
2,Asha,CSE
3,Kiran,NaN
4,Meena,CSE
5,Priya,CSE
6,Ravi,ECE


## 14. 14. What is an INNER JOIN?

It returns rows that have matching values in both joined tables.

In [14]:
run("""
SELECT e.name, d.dept_name
FROM Employees e
INNER JOIN Departments d ON e.dept_id = d.dept_id
""")   # Kiran (no dept) and MECH (no employees) are not in the result

,name,dept_name
0,Asha,CSE
1,Ravi,ECE
2,Priya,CSE
3,Arun,EEE
4,Meena,CSE


## 15. 15. What is an index?

An index is a database structure that can improve the speed of data retrieval, at the cost of additional storage and maintenance.

In [15]:
execute("CREATE INDEX IF NOT EXISTS idx_emp_salary ON Employees(salary)")

plan = run("EXPLAIN QUERY PLAN SELECT * FROM Employees WHERE salary = 60000")
print("Query plan (note it uses the index):")
plan

Query plan (note it uses the index):


,id,parent,notused,detail
0,3,0,0,SEARCH Employees USING INDEX idx_emp_salary (s...


## 16. 16. What is a transaction?

A transaction is a logical unit of database operations that should be completed successfully as a whole.

In [16]:
execute("DROP TABLE IF EXISTS Accounts; CREATE TABLE Accounts (name TEXT, balance INTEGER)")
conn.execute("INSERT INTO Accounts VALUES ('A', 1000), ('B', 500)")
conn.commit()

# Transfer 200 from A to B as one transaction
conn.execute("UPDATE Accounts SET balance = balance - 200 WHERE name = 'A'")
conn.execute("UPDATE Accounts SET balance = balance + 200 WHERE name = 'B'")
conn.commit()
print("After successful transfer:")
display(run("SELECT * FROM Accounts"))

# A failed transaction is rolled back completely
conn.execute("UPDATE Accounts SET balance = balance - 999 WHERE name = 'A'")
conn.rollback()
print("After rollback (the -999 change is undone):")
run("SELECT * FROM Accounts")

After successful transfer:


,name,balance
0,A,800
1,B,700


After rollback (the -999 change is undone):


,name,balance
0,A,800
1,B,700


## 17. 17. What are ACID properties?

- Atomicity
- Consistency
- Isolation
- Durability

In [17]:
execute("DROP TABLE IF EXISTS Bank; CREATE TABLE Bank (name TEXT PRIMARY KEY, balance INTEGER CHECK (balance >= 0))")
conn.execute("INSERT INTO Bank VALUES ('A', 100), ('B', 100)")
conn.commit()

# Atomicity + Consistency: second step violates CHECK, so the whole transaction is undone
try:
    conn.execute("UPDATE Bank SET balance = balance + 150 WHERE name = 'B'")   # step 1 works
    conn.execute("UPDATE Bank SET balance = balance - 150 WHERE name = 'A'")   # step 2 fails (negative)
    conn.commit()
except sqlite3.IntegrityError as e:
    conn.rollback()
    print("Transaction failed and was rolled back ->", e)

print("Balances unchanged (Atomicity & Consistency):")
run("SELECT * FROM Bank")
# Isolation: concurrent transactions do not see each other's uncommitted changes.
# Durability: once committed, data survives crashes (stored on disk).

Transaction failed and was rolled back -> CHECK constraint failed: balance >= 0
Balances unchanged (Atomicity & Consistency):


,name,balance
0,A,100
1,B,100


## 18. 18. What is a view?

A view is a virtual table based on the result of a query.

In [18]:
execute("""
DROP VIEW IF EXISTS HighPaid;
CREATE VIEW HighPaid AS
SELECT name, salary FROM Employees WHERE salary >= 50000;
""")
run("SELECT * FROM HighPaid")

,name,salary
0,Asha,50000.0
1,Priya,60000.0
2,Meena,60000.0


## 19. 19. What is a subquery?

A subquery is a query written inside another SQL query.

In [19]:
print("Employees earning more than the average salary:")
run("""
SELECT name, salary FROM Employees
WHERE salary > (SELECT AVG(salary) FROM Employees)
""")

Employees earning more than the average salary:


,name,salary
0,Asha,50000.0
1,Priya,60000.0
2,Meena,60000.0


## 20. 20. What is the difference between WHERE and HAVING?

`WHERE` filters rows before grouping, while `HAVING` filters groups after `GROUP BY`.

In [20]:
print("WHERE filters rows BEFORE grouping:")
display(run("""
SELECT dept_id, COUNT(*) AS emp_count
FROM Employees
WHERE salary > 35000
GROUP BY dept_id
"""))

print("HAVING filters groups AFTER grouping:")
run("""
SELECT dept_id, COUNT(*) AS emp_count
FROM Employees
GROUP BY dept_id
HAVING COUNT(*) > 1
""")

WHERE filters rows BEFORE grouping:


,dept_id,emp_count
0,1,3
1,2,1
2,3,1


HAVING filters groups AFTER grouping:


,dept_id,emp_count
0,1,3
